<a href="https://colab.research.google.com/github/vickytoriag/Home-Work/blob/main/Final_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os

os.environ["VULNERS_API_KEY"] = "ваш_ключ"

In [7]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt

# ==========================
# НАСТРОЙКИ
# ==========================

LOG_FILE = "alerts-only.json"

CVSS_LIMIT = 7.0
IP_LIMIT = 3

API_KEY = os.getenv("VULNERS_API_KEY")

TELEGRAM_TOKEN = os.getenv("TELEGRAM_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

SEARCH_KEYWORDS = [
    "ssh",
    "vnc",
    "mssql",
    "oracle",
    "mysql",
    "postgresql",
    "snmp",
    "sip"
]


# ==========================
# ЗАГРУЗКА ЛОГОВ SURICATA
# ==========================

def load_suricata_logs(filename: str) -> list:

    if not os.path.exists(filename):
        print(f"❌ Файл {filename} не найден.")
        return []

    try:
        with open(filename, "r", encoding="utf-8") as f:
            content = f.read().strip()

        if not content:
            print(f"❌ Файл {filename} пуст.")
            return []

        # Если начинается с [, читаем как JSON
        if content.startswith("["):
            data = json.loads(content)
            if isinstance(data, list):
                return data
            print("❌ Ошибка чтения логов.")
            return []

        # Иначе пытаемся читать как NDJSON
        logs = []
        for line in content.splitlines():
            line = line.strip()
            if line:
                logs.append(json.loads(line))
        return logs

    except Exception as e:
        print(f"❌ Ошибка чтения логов: {e}")
        return []


# ==========================
# TELEGRAM
# ==========================

def notify_telegram(message: str) -> None:
    """
    Отправляет уведомление в Telegram.
    Если TELEGRAM_TOKEN или TELEGRAM_CHAT_ID не заданы,
    выполняется имитация отправки в консоль.
    """
    if not TELEGRAM_TOKEN or not TELEGRAM_CHAT_ID:
        print("\n[ИМИТАЦИЯ TELEGRAM]")
        print(message)
        print()
        return

    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message
    }

    try:
        response = requests.post(url, json=payload, timeout=10)
        if response.status_code == 200:
            print("[TELEGRAM] Notification sent")
        else:
            print("[TELEGRAM] Notification failed:", response.status_code)
    except Exception as e:
        print("[TELEGRAM] Notification error:", e)


# ==========================
# API VULNERS
# ==========================

def check_vulnerabilities(keyword: str) -> list:
    """
    Поиск уязвимостей по ключевому слову,
    возвращаются только те, у которых CVSS >= CVSS_LIMIT
    """
    if not API_KEY:
        print("❌ Не задан VULNERS_API_KEY")
        return []

    url = "https://vulners.com/api/v3/search/lucene/"
    headers = {"X-Api-Key": API_KEY}
    params = {
        "query": keyword,
        "size": 10
    }

    try:
        response = requests.get(url, headers=headers, params=params, timeout=15)
        if response.status_code != 200:
            print(f"Ошибка Vulners для '{keyword}': {response.status_code}")
            return []

        data = response.json()
        critical = []

        for item in data.get("data", {}).get("search", []):
            source = item.get("_source", {})
            cvss = source.get("cvss", {}).get("score", 0)
            vuln_id = source.get("id")

            if cvss >= CVSS_LIMIT and vuln_id:
                critical.append({
                    "keyword": keyword,
                    "id": vuln_id,
                    "cvss": cvss,
                    "title": source.get("title")
                })

        return critical

    except Exception as e:
        print(f"Ошибка запроса Vulners для '{keyword}': {e}")
        return []


def collect_all_vulnerabilities() -> list:
    """
    Проверяет ключевые слова и собирает список критичных уязвимостей.
    """
    all_vulns = []

    print("\n🔎 Поиск критичных уязвимостей через Vulners:\n")

    for keyword in SEARCH_KEYWORDS:
        vulns = check_vulnerabilities(keyword)

        if vulns:
            print(f"[{keyword}] найдено критичных уязвимостей: {len(vulns)}")
            for vuln in vulns:
                print(f"ID: {vuln['id']}")
                print(f"CVSS: {vuln['cvss']}")
                print(f"Название: {vuln['title']}")
                print("-" * 40)

        all_vulns.extend(vulns)

    unique = {}
    for vuln in all_vulns:
        if vuln.get("id"):
            unique[vuln["id"]] = vuln

    return list(unique.values())


# ==========================
# АНАЛИЗ ЛОГОВ SURICATA
# ==========================

def analyze_logs(filename: str):
    """
    Загружает alerts-only.json и выполняет анализ.
    """
    logs = load_suricata_logs(filename)
    if not logs:
        return pd.DataFrame(), pd.Series(dtype=int), pd.Series(dtype=int)

    df = pd.json_normalize(logs)

    if "src_ip" not in df.columns:
        print("❌ Поле src_ip отсутствует в логах.")
        return pd.DataFrame(), pd.Series(dtype=int), pd.Series(dtype=int)

    if "alert.signature" not in df.columns:
        print("❌ Поле alert.signature отсутствует в логах.")
        return pd.DataFrame(), pd.Series(dtype=int), pd.Series(dtype=int)

    print("\nЗагружено событий:", len(df))
    print("Уникальных IP:", df["src_ip"].nunique())

    ip_counts = df["src_ip"].value_counts()
    signature_counts = df["alert.signature"].value_counts()

    print("\n🚩 Подозрительные IP:\n")
    suspicious_ips = ip_counts[ip_counts >= IP_LIMIT]
    if suspicious_ips.empty:
        print("Подозрительных IP не найдено по текущему порогу.")
    else:
        for ip, count in suspicious_ips.items():
            print(f"{ip}: {count} событий")

    print("\n📡 Частые сигнатуры атак:\n")
    print(signature_counts.head())

    return df, ip_counts, signature_counts


# ==========================
# РЕАГИРОВАНИЕ
# ==========================

def respond_to_threats(df: pd.DataFrame, ip_counts: pd.Series) -> pd.DataFrame:
    """
    Имитирует блокировку подозрительных IP и формирует таблицу результатов.
    """
    columns = ["ip", "events", "top_signature", "is_threat"]

    if df.empty or ip_counts.empty:
        return pd.DataFrame(columns=columns)

    suspicious_ips = ip_counts[ip_counts >= IP_LIMIT]
    rows = []

    for ip, count in suspicious_ips.items():
        ip_signatures = df[df["src_ip"] == ip]["alert.signature"].value_counts()
        top_signature = ip_signatures.index[0] if len(ip_signatures) > 0 else None

        print(f"\n[ACTION] Имитация блокировки IP {ip}")

        notify_telegram(
            f"🚨 КРИТИЧЕСКАЯ УГРОЗА ОБНАРУЖЕНА!\n"
            f"Источник: Suricata logs\n"
            f"IP: {ip}\n"
            f"Количество событий: {count}\n"
            f"Сигнатура: {top_signature}\n"
            f"Действие: имитация блокировки IP"
        )

        rows.append({
            "ip": ip,
            "events": int(count),
            "top_signature": top_signature,
            "is_threat": True
        })

    if not rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows)


# ==========================
# ОТЧЕТ
# ==========================

def save_report(report_df: pd.DataFrame,
                signature_counts: pd.Series,
                vulnerabilities: list) -> None:
    """
    Сохраняет JSON и CSV отчет.
    """
    threats_found = int(report_df["is_threat"].sum()) if "is_threat" in report_df.columns else 0

    report_json = {
        "summary": {
            "checked_ips": int(len(report_df)),
            "threats_found": threats_found,
            "critical_vulnerabilities_found": len(vulnerabilities)
        },
        "top_attack_signatures": signature_counts.head(5).to_dict() if not signature_counts.empty else {},
        "critical_vulnerabilities": vulnerabilities,
        "ip_results": report_df.to_dict(orient="records")
    }

    with open("report.json", "w", encoding="utf-8") as f:
        json.dump(report_json, f, indent=2, ensure_ascii=False)

    report_df.to_csv("ip_activity.csv", index=False)

    print("\nОтчёт сохранён:")
    print("report.json")
    print("ip_activity.csv")


# ==========================
# ВИЗУАЛИЗАЦИЯ
# ==========================

def plot_graph(ip_counts: pd.Series) -> None:
    """
    Строит график топ-5 IP по количеству событий.
    """
    if ip_counts.empty:
        print("Нет данных для графика")
        return

    top_ips = ip_counts.head(5)

    plt.figure(figsize=(8, 5))
    plt.bar(top_ips.index, top_ips.values)

    plt.title("Top Suspicious IP Activity")
    plt.xlabel("IP address")
    plt.ylabel("Alert count")

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("ip_activity.png")
    plt.close()

    print("График сохранён: ip_activity.png")


# ==========================
# ГЛАВНАЯ ЧАСТЬ
# ==========================

if __name__ == "__main__":

    print("Запуск мониторинга угроз\n")

    vulnerabilities = collect_all_vulnerabilities()

    df, ip_counts, signature_counts = analyze_logs(LOG_FILE)

    if not df.empty:
        report_df = respond_to_threats(df, ip_counts)
        save_report(report_df, signature_counts, vulnerabilities)
        plot_graph(ip_counts)

    print("\nАнализ завершён")

Запуск мониторинга угроз


🔎 Поиск критичных уязвимостей через Vulners:

[ssh] найдено критичных уязвимостей: 4
ID: CVE-1999-0013
CVSS: 8.4
Название: CVE-1999-0013
----------------------------------------
ID: CVE-2001-1473
CVSS: 7.5
Название: CVE-2001-1473
----------------------------------------
ID: CVE-2001-1476
CVSS: 7.5
Название: CVE-2001-1476
----------------------------------------
ID: CVE-2001-1475
CVSS: 7.5
Название: CVE-2001-1475
----------------------------------------
[vnc] найдено критичных уязвимостей: 5
ID: VERACODE:23806
CVSS: 10.0
Название: Arbitrary Code Execution
----------------------------------------
ID: PRION:CVE-2017-5885
CVSS: 7.5
Название: Integer overflow
----------------------------------------
ID: CVE-2017-1000044
CVSS: 9.8
Название: CVE-2017-1000044
----------------------------------------
ID: CVE-2006-2369
CVSS: 7.5
Название: CVE-2006-2369
----------------------------------------
ID: PRION:CVE-2017-1000044
CVSS: 7.5
Название: Memory corruption
------------